## Agen란?
- LLM이 스스로 판단해서 어떤 행동(툴 사용 포함)을 할 지 결정하는 실행주체를 의미합니다.

In [1]:
from dotenv import load_dotenv
from langchain_openai.chat_models.base import ChatOpenAI
import os

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-_2Lh2bEy5isJ


In [2]:
prompt = "cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12"

In [2]:
# chat = ChatOpenAI(model="gpt-4o", temperature=0.1)
chat = ChatOpenAI(temperature=0.1)

In [7]:
result = chat.invoke(prompt)

In [9]:
print(result.content)

To find the total cost, you need to add all the amounts together:

\[ 
355.39 + 924.87 + 721.2 + 1940.29 + 573.63 + 65.72 + 35.00 + 522.00 + 76.16 + 29.12 
\]

Calculating step-by-step:

1. \( 355.39 + 924.87 = 1280.26 \)
2. \( 1280.26 + 721.2 = 2001.46 \)
3. \( 2001.46 + 1940.29 = 3941.75 \)
4. \( 3941.75 + 573.63 = 4515.38 \)
5. \( 4515.38 + 65.72 = 4581.10 \)
6. \( 4581.10 + 35.00 = 4616.10 \)
7. \( 4616.10 + 522.00 = 5138.10 \)
8. \( 5138.10 + 76.16 = 5214.26 \)
9. \( 5214.26 + 29.12 = 5243.38 \)

The total cost is \( \boxed{5243.38} \).


In [11]:
"""
    프롬프트의 정답
    $4,363.38.

    계산기에서 직접 계산하기 ↓↓↓
    $5,243.38

    llm의 계산 착오
    ※이유※
    LLM은 산술 연산을 수행하지 않습니다. 이런 계산은 AI보다 계산기가 더 잘합니다.
    LLM은 text를 생성해내는 모델입니다. 문장의 시퀀스의 다음 token이 무엇인지 통계적으로 추측합니다.
    이러한 LLm의 오류를 잡기 위해서는 agent를 제공해주어야 합니다.
    그리고 agent를 위한 tool(툴)을 만들고, agent가 tool을 선택해서 실행하는 것입니다.
"""

'\n    프롬프트의 정답\n    $4,363.38.\n\n    계산기에서 직접 계산하기 ↓↓↓\n    $5,243.38\n\n    llm의 계산 착오\n    ※이유※\n    LLM은 산술 연산을 수행하지 않습니다. 이런 계산은 AI보다 계산기가 더 잘합니다.\n    LLM은 text를 생성해내는 모델입니다. 문장의 시퀀스의 다음 token이 무엇인지 통계적으로 추측합니다.\n    이러한 LLm의 오류를 잡기 위해서는 agent를 제공해주어야 합니다.\n    그리고 agent를 위한 tool(툴)을 만들고, agent가 tool을 선택해서 실행하는 것입니다.\n'

## Agent 생성

In [3]:
# create_agent 함수 통일
from langchain.agents import create_agent
from langchain.tools import tool

In [14]:
@tool
def plus(num1: float, num2: float) -> float:
    """
        Adds two numbers and return the result.
    """
    return num1 + num2


"""
    json {
        "name": plus,
        "description": "Adds two numbers and return the result.",
        "parameters": {
            "num1": "float",
            "num2": "float",
        }
    
    }

"""

None

In [15]:
agent = create_agent(
    model="gpt-3.5-turbo",
    tools=[plus],
    system_prompt="You are a helpful assistant"
)

In [16]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": prompt
        }
    ]
})

In [19]:
# result
for message in result["messages"]:
    if message.__class__.__name__ == "AIMessage" and message.tool_calls:
        for i in message.tool_calls:
            print(i)

{'name': 'plus', 'args': {'num1': 355.39, 'num2': 924.87}, 'id': 'call_EPGhGg5b7NiatlaYPy6lHPcr', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 721.2, 'num2': 1940.29}, 'id': 'call_7OddYzIAshBOPKKzIjLRxZ5C', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 573.63, 'num2': 65.72}, 'id': 'call_PGi1dn2j1DxhnXrTH8nKYm1H', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 35.0, 'num2': 522.0}, 'id': 'call_fGr0MOzk9zDdGQMWlYeoOpSP', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 76.16, 'num2': 29.12}, 'id': 'call_3LyYmlUeNOk9eV4Mn4wtTcSX', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 2661.49, 'num2': 105.28}, 'id': 'call_kW8BKGOv38SATEErCSCyGD1C', 'type': 'tool_call'}


In [6]:
@tool
def total_sum(numbers: list[float]) -> float:
    """
        Adds a list of numbers and returns the total sum.
        Use this tool when you need to calculate the total of multiple numbers.
        Input should be a string representation of a list.
        Example: "[1, 2, 3]"
    """

    return sum(numbers)

In [7]:
agent = create_agent(
    model="gpt-3.5-turbo",
    tools=[total_sum],
    system_prompt="You are a helpful assistant"
)

In [8]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": prompt
        }
    ]
})

In [9]:
result["messages"][-1].content

'The total sum of $355.39 + $924.87 + $721.20 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12 is $5243.38.'

In [24]:
for message in result["messages"]:
    if message.__class__.__name__ == "AIMessage" and message.tool_calls:
        for i in message.tool_calls:
            print(i)

{'name': 'total_sum', 'args': {'numbers': [355.39, 924.87, 721.2, 1940.29, 573.63, 65.72, 35, 522, 76.16, 29.12]}, 'id': 'call_1DWdrPrVqnNy2GNr1fYM0djg', 'type': 'tool_call'}


### 랭스미스

# LangSmith(랭스미스)
- LLM 기반 애플리케이션의 디버깅, 성능 평가, 모니터링 등을 제공하는 랭체인의 통합 플랫폼입니다.

## 랭스미스 Open API Key 발급
- https://smith.langchain.com/ 접속
- 로그인 후 좌측 하단 [Setting] 메뉴 클릭
- [API Keys] 클릭 후 생성
- Description은 lang_ksh(이니셜)
- [default workspace]는 기존에 있는 workspace1로 만들고 생성
- 발급받은 KEY를 .env에 추가하기

### .env에 추가하기
- LANGCHAIN_TRACING_V2=true
- LANGCHAIN_ENDPOINT="https://api.smith.langchain.com"
- LANGCHAIN_PROJECT=lang_1900
- LANGSMITH_API_KEY=발급받은 랭스미스 key

### 설정 후 Jupyter Notebook 재실행

## Agent가 동작하는 과정
1. 끝날때까지 반복이 되는 loop입니다.
2. llm으로 부터 어떤 것을 할지(get action)을 받아온다. (lang smith의 output에서 확인가능)
3. 실행한 결과를 observation이라고 부른다. 다시 다음 next action을 실행시킨다.
4. Agent Finish를 응답받으면 마지막 action 값을 리턴한다.

## 1. ReAct (Reasoning and Acting) Agent

In [4]:
from dotenv import load_dotenv
import os

from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool, BaseTool
from langchain_classic.agents import AgentExecutor, create_react_agent, create_openai_functions_agent
from langchain_classic import hub
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List #Python의 내장 모듈 typing

In [5]:
load_dotenv()

True

In [6]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [13]:
@tool
def plus(expression: str) -> float:
    """
        Adds multiple numbers and returns their total sum.

        The input must be a comma-spreated string of numbers.
        Example: "10,20,30"

        Use this tool when you need to calculate the sum of multiple values.
    """
    try:
        numbers = [float(num) for num in expressionq.split(",")]
        return sum(numbers)
    except Exception as e:
        return -1

In [14]:
tools = [plus]
react_agent_prompt = hub.pull("hwchase17/react")

# 판단
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt = react_agent_prompt
)

# 실행기
react_agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True # 내부 동작 확인
)

In [16]:
prompt = "cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12"
result = react_agent_executor.invoke({
    "input": prompt
})



> Entering new AgentExecutor chain...
To find the total cost, I need to sum all the given amounts. I will use the plus function to calculate the total.

Action: plus
Action Input: "355.39,924.87,721.2,1940.29,573.63,65.72,35.00,522.00,76.16,29.12"-1It seems there was an issue with the calculation. I will double-check the input format and try again to ensure that the numbers are correctly formatted for the plus function.

Action: plus
Action Input: "355.39,924.87,721.2,1940.29,573.63,65.72,35.00,522.00,76.16,29.12"-1It appears that the plus function is not accepting the input as expected. I will check if there are any formatting issues or if the function has limitations on the number of decimal places or the total number of inputs. 

Since the input seems correct, I will try simplifying the input by rounding the numbers to two decimal places and ensuring there are no extra spaces.

Action: plus
Action Input: "355.39,924.87,721.20,1940.29,573.63,65.72,35.00,522.00,76.16,29.12"-1It seem

## 2. OpenAI Function Calling Agent

In [7]:
class CalculatorToolArgsSchema(BaseModel):
    numbers: List[float] = Field(description="Numbers to sum")

class CalculatorTool(BaseTool):
    # 약속된 필드 이름 (공백x, 한글x, a-z, A-Z, 0-9, _, -만 가능) 
    name: Type[str] = "calculator_tool"
    description: Type[str] = """
        Adds multiple numbers and returns their total sum.
        Use this tools when you need to calculator the sum of multiple values.
    """

    args_schema: Type[BaseModel] = CalculatorToolArgsSchema

    # BaseTool은 반드시 _run 함수를 재정의
    # tool을 호출했을 때 실행되는 메인로직
    def _run(self, numbers):
        return sum(numbers)

In [8]:
tools = [CalculatorTool()]

# placeholder(agent_scratchpad): 내부 tool, reasoning 호출 기록을 임시 저장
function_agent_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("placeholder", """{agent_scratchpad}"""),
])

# 판단
agent = create_openai_functions_agent(
    llm=llm,
    tools=tools,
    prompt=function_agent_prompt
)

# 실행기
calling_agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [23]:
prompt = "cost of $455.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $39.12"
result = calling_agent_executor.invoke({
    "input": prompt
})



> Entering new AgentExecutor chain...

Invoking: `calculator_tool` with `{'numbers': [455.39, 924.87, 721.2, 1940.29, 573.63, 65.72, 35, 522, 76.16, 39.12]}`


5353.38The total cost is $5,353.38.

> Finished chain.


In [24]:
result["output"]

'The total cost is $5,353.38.'

# v1.0이상 create_agent(커스텀 툴)

In [34]:
from dotenv import load_dotenv
import os
import requests

from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool, BaseTool
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List, Tuple, Dict #Python의 내장 모듈 typing

# 추가된 import
from langchain_community.utilities.duckduckgo_search import DuckDuckGoSearchAPIWrapper
from geopy.geocoders import Nominatim

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-_2Lh2bEy5isJ


In [11]:
tools = []

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

In [12]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "강남의 현재 실시간 날씨 알려줘!")
])

chain = prompt | agent
result = chain.invoke({})

result

{'messages': [HumanMessage(content='강남의 현재 실시간 날씨 알려줘!', additional_kwargs={}, response_metadata={}, id='5ebb71b0-b07d-4a86-8d06-c4d33933d79c'),
  AIMessage(content='죄송하지만, 현재 실시간 날씨 정보를 제공할 수는 없습니다. 그러나 강남의 날씨를 확인하려면 기상청 웹사이트나 날씨 앱을 참고하시면 정확한 정보를 얻을 수 있습니다. 혹시 다른 질문이 있으시면 도와드리겠습니다!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 18, 'total_tokens': 78, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_d2f048fbf1', 'id': 'chatcmpl-Dh6znMCDoHPtqkU9JLESmWpqPRLux', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e3ea3-9dce-7ce1-8569-e31923867691-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 

In [24]:
result["messages"][-1].content

'죄송하지만, 현재 실시간 날씨 정보를 제공할 수는 없습니다. 그러나 강남의 날씨를 확인하려면 기상청 웹사이트나 날씨 앱을 참고하시면 정확한 정보를 얻을 수 있습니다. 혹시 다른 질문이 있으시면 도와드리겠습니다!'

In [31]:
#지역 => 위도, 경도
def get_coordinates(location_name):
    locator = Nominatim(user_agent="URA")
    location = locator.geocode(location_name)

    return location.latitude, location.longitude
    # print(location)
    # print(location.longitude)
    # print(location.latitude)

In [32]:
get_coordinates("강남")

(37.4979497, 127.0275574)

In [46]:
# 위도, 경도 => 날씨
def get_weather(lat, lon):
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"

    # 기상 코드(WMO Code)를 한글로 변환하는 딕셔너리
    weather_codes = {
        0: "맑음 ☀️",
        1: "대체로 맑음 🌤️", 2: "구름 조금 ⛅", 3: "흐림 ☁️",
        45: "안개 🌫️", 48: "침강 안개 🌫️",
        51: "가벼운 이슬비 🌦️", 53: "이슬비 🌧️", 55: "강한 이슬비 ⛈️",
        61: "약한 비 💧", 63: "보통 비 ☔", 65: "강한 비 🌊",
        71: "약한 눈 ❄️", 73: "보통 눈 ☃️", 75: "강한 눈 🏔️",
        80: "약한 소나기 🌦️", 81: "보통 소나기 🌧️", 82: "강한 소나기 ⛈️",
        95: "뇌우 ⚡", 96: "뇌우 및 우박 ⛈️", 99: "심한 뇌우 🌪️"
    }
    
    try:
        response = requests.get(url)
        datas = response.json()

        if "current_weather" in datas:
            current = datas["current_weather"]
            temp = current["temperature"]
            wind = current["windspeed"]
            code = current["weathercode"]

            condition = weather_codes.get(code, "알 수 없음")

            return f"상태: {condition}\n온도: {temp}°C\n풍속: {wind}km/h"
        
    except Exception as e:
        return "요청 실패"

In [23]:
#37.500078, 127.035548
print(get_weather(37.500078, 127.035548))

상태: 흐림 ☁️
온도: 23.1°C
풍속: 5.4km/h


## 함수 => 툴로 변경 후 제공

In [47]:
class CoordinatesToolArgSchema(BaseModel):
    location_name: str = Field("위도와 경도로 바꾸고 싶은 장소명입니다.")

class CoordinatesTool(BaseTool):
    name: Type[str] = "coordinates_tool"
    description: Type[str] = """
        장소명을 위도(latitude)와 경도(longitude) 좌표로 변환합니다.
        장소명을 위도와 경도로 변환하고 싶을 대 사용하는 도구입니다.
    """
    args_schema: Type[BaseModel] = CoordinatesToolArgSchema

    def _run(self, location_name: str) -> Tuple[float, float]:
        locator = Nominatim(user_agent="ksh")
        location = locator.geocode(location_name)
    
        return location.latitude, location.longitude, 

In [38]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "강남의 위도와 경도를 알려줘")
])

chain = prompt | agent
result = chain.invoke({})

result

{'messages': [HumanMessage(content='강남의 위도와 경도를 알려줘', additional_kwargs={}, response_metadata={}, id='8f4205dc-d9b9-40e3-b632-cce141c29804'),
  AIMessage(content='강남구의 위도와 경도는 대략 다음과 같습니다:\n\n- 위도: 37.4979° N\n- 경도: 127.0276° E\n\n강남구는 서울의 중요한 상업 및 주거 지역 중 하나입니다. 구체적인 위치는 조금씩 다를 수 있으니, 특정 장소에 대한 위도와 경도를 알고 싶다면 해당 장소를 추가로 알려주시면 더 정확히 안내해 드리겠습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 17, 'total_tokens': 116, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_2cb7f3c26f', 'id': 'chatcmpl-Dh7U2nO77oUnFNFTCJ2xd79oxtlmu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e3ec0-376c-7522-bb3d-3ad7a14f492e-0', tool_calls=[], invalid_tool

In [48]:
class WeatherSearchToolArgSchema(BaseModel):
    lat: float = Field(description="위도, Example Value: 37.500078")
    lon: float = Field(description="경도, Example Value: 127.035548")
    
class WeatherSearchTool(BaseTool):
    name: Type[str] = "weather_search_tool"
    description: Type[str] = """
        지역의 날씨를 가져오고 싶을 때 사용하는 툴입니다.
        위도와 경도를 입력하면, 해당 지역의 날씨의 정보를 문자열로 반환합니다.
    """

    args_schema: Type[BaseModel] = WeatherSearchToolArgSchema
    
    # 위도, 경도 -> 날씨
    def _run(self, lat: float, lon: float) -> str:
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        
        # 기상 코드(WMO Code)를 한글로 변환하는 딕셔너리
        weather_codes = {
            0: "맑음 ☀️",
            1: "대체로 맑음 🌤️", 2: "구름 조금 ⛅", 3: "흐림 ☁️",
            45: "안개 🌫️", 48: "침강 안개 🌫️",
            51: "가벼운 이슬비 🌦️", 53: "이슬비 🌧️", 55: "강한 이슬비 ⛈️",
            61: "약한 비 💧", 63: "보통 비 ☔", 65: "강한 비 🌊",
            71: "약한 눈 ❄️", 73: "보통 눈 ☃️", 75: "강한 눈 🏔️",
            80: "약한 소나기 🌦️", 81: "보통 소나기 🌧️", 82: "강한 소나기 ⛈️",
            95: "뇌우 ⚡", 96: "뇌우 및 우박 ⛈️", 99: "심한 뇌우 🌪️"
        }
    
        try:
            response = requests.get(url)
            datas = response.json()
    
            if "current_weather" in datas:
                current = datas["current_weather"]
                temp = current["temperature"]
                wind = current["windspeed"]
                code = current["weathercode"]
    
                condition = weather_codes.get(code, "알 수 없음")
    
                return f"상태: {condition}\n온도: {temp}°C\n풍속: {wind}km/h"
            
        except Exception as e:
            return "요청 실패"

In [49]:
tools = [CoordinatesTool(), WeatherSearchTool()]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "강남의 현재 실시간 날씨 알려줘!")
])

chain = prompt | agent
result = chain.invoke({})

result


{'messages': [HumanMessage(content='강남의 현재 실시간 날씨 알려줘!', additional_kwargs={}, response_metadata={}, id='d9deb214-11f3-49a5-a2bc-207e64c81845'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 185, 'total_tokens': 201, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_da1f8e43b9', 'id': 'chatcmpl-Dh7rCkNSEduX9TWxqNgUxZhjP4e5R', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e3ed6-229b-7ad1-943a-fa4d26ef444c-0', tool_calls=[{'name': 'coordinates_tool', 'args': {'location_name': '강남'}, 'id': 'call_uHDdfzsdddU977ETkZQ2Lyx3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 185, 'out

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "강남의 현재 실시간 날씨 알려줘!")
])

chain = prompt | agent
result = chain.invoke({})

result

## Stock Agent

In [18]:
from dotenv import load_dotenv
import os
import requests

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain.tools import tool, BaseTool
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List, Tuple, Dict #Python의 내장 모듈 typing

# 추가된 import
from langchain_community.utilities.duckduckgo_search import DuckDuckGoSearchAPIWrapper
from geopy.geocoders import Nominatim

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-_2Lh2bEy5isJ


## 1. 일, 주, 월 단위 실적을 제공
- https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=IBM&apikey=demo

In [3]:
"""
    "Meta Data": {
        "1. Information": "Daily Prices (open, high, low, close) and Volumes",
        "2. Symbol": "IBM",
        "3. Last Refreshed": "2026-05-18",
        "4. Output Size": "Compact",
        "5. Time Zone": "US/Eastern"
    },
    "Time Series (Daily)": {
        "2026-05-18": {
            "1. open": "218.5500", #시작 가격
            "2. high": "223.3300", #가장 최고가
            "3. low": "217.7500", #가장 최저가
            "4. close": "222.7500", #마감가
            "5. volume": "5946367" #거래량
        },
        "2026-05-15": {
            "1. open": "218.2000",
            "2. high": "220.9100",
            "3. low": "217.6150",
            "4. close": "219.3000",
            "5. volume": "6154450"
        },
"""

None

## 2. News & Sentiments (최신뉴스 & 민감도)
- https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers=AAPL&apikey=demo

In [5]:
"""
{
    "items": "50",
    # 뉴스의 지표
    "sentiment_score_definition": "x <= -0.35: Bearish; -0.35 < x <= -0.15: Somewhat-Bearish; -0.15 < x < 0.15: Neutral; 0.15 <= x < 0.35: Somewhat_Bullish; x >= 0.35: Bullish",
    "relevance_score_definition": "0 < x <= 1, with a higher score indicating higher relevance.",
    "feed": [
        {
            "title": "Apple CEO Tim Cook in Beijing with US Presidential Delegation – May 2026 - News and Statistics",
            "url": "https://www.indexbox.io/blog/tim-cook-joins-us-delegation-to-beijing-as-apple-navigates-china-ties/",
            "time_published": "20260519T031958",
            "authors": [],
            "summary": "Apple CEO Tim Cook is in Beijing as part of a U.S. presidential delegation, marking the first such visit in nearly a decade. This trip is crucial for Apple due to its extensive manufacturing operations in China and China being its largest market outside the U.S. Improved U.S.-China relations, including reduced tariffs and increased market access, would significantly benefit Apple, especially following Chinese President Xi Jinping's recent pledge to \"open wider\" for American businesses.",
            "banner_image": "https://www.indexbox.io/landing/img/blog/telegram-fallback/5eab958188a1a3d707e92b07af013cb6.webp",
            "source": "IndexBox",
            "category_within_source": "General",
            "source_domain": "IndexBox",
            "topics": [
                {
                    "topic": "technology",
                    "relevance_score": "0.812212"
                },
                {
                    "topic": "economy_macro",
                    "relevance_score": "0.745762"
                },
                {
                    "topic": "finance",
                    "relevance_score": "0.634877"
                },
                {
                    "topic": "manufacturing",
                    "relevance_score": "0.647957"
                }
            ],
            # 0.335609: 뉴스 지표 점수(Somewhat_Bullish)
            "overall_sentiment_score": 0.335609,
            "overall_sentiment_label": "Somewhat-Bullish",
            "ticker_sentiment": [
                {
                    "ticker": "AAPL",
                    "relevance_score": "1.000000",
                    "ticker_sentiment_score": "0.423983",
                    "ticker_sentiment_label": "Bullish"
                },
                {
                    "ticker": "NVDA",
                    "relevance_score": "0.610471",
                    "ticker_sentiment_score": "0.315554",
                    "ticker_sentiment_label": "Somewhat-Bullish"
                },
                {
                    "ticker": "TSLA",
                    "relevance_score": "0.611822",
                    "ticker_sentiment_score": "0.327856",
                    "ticker_sentiment_label": "Somewhat-Bullish"
                }
            ]
        },
"""

None

## 3. 회사 재무 재표, 손익(Income Statement)
- https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol=IBM&apikey=demo

In [7]:
"""
{
    "symbol": "IBM",
    "annualReports": [
        {
            "fiscalDateEnding": "2025-12-31",
            "reportedCurrency": "USD",
            "grossProfit": "40185000000",   #순수익
            "totalRevenue": "67535000000",    #총매출
            "costOfRevenue": "27350000000",    #원가
            "costofGoodsAndServicesSold": "27350000000",
            "operatingIncome": "10325000000",
            "sellingGeneralAndAdministrative": "18285000000",
            "researchAndDevelopment": "8320000000",
            "operatingExpenses": "29860000000",
            "investmentIncomeNet": "None",
            "netInterestIncome": "-1290000000",
            "interestIncome": "645000000",
            "interestExpense": "1935000000",
            "nonInterestIncome": "None",
            "otherNonOperatingIncome": "None",
            "depreciation": "None",
            "depreciationAndAmortization": "5021000000",
            "incomeBeforeTax": "10328000000",
            "incomeTaxExpense": "-242000000",
            "interestAndDebtExpense": "None",
            "netIncomeFromContinuingOperations": "10571000000",
            "comprehensiveIncomeNetOfTax": "None",
            "ebit": "12263000000",
            "ebitda": "17284000000",
            "netIncome": "10593000000"
        },
"""

None

## 4. 회사의 개요(Company OverView)
- https://www.alphavantage.co/query?function=OVERVIEW&symbol=IBM&apikey=demo

## Stock Tools 생성

In [27]:
class StockMartSymbolSearchToolArgSchema(BaseModel):
    query: str = Field(description="""
        The query you will search for Example query: Stock Market Symbol for Apple company.
    """)

class StockMartSymbolSearchTool(BaseTool):
    name: Type[str] = "stock_mark_symbol_search_tool"
    description: Type[str] = """
        Use this tool fin the stock market symbol for a company.
        It takes a query as an argument.
    """

    args_schema: Type[BaseModel] = StockMartSymbolSearchToolArgSchema
    
    def _run(self, query):
        ddg = DuckDuckGoSearchAPIWrapper()
        ddg.run(query)

In [13]:
ddg = DuckDuckGoSearchAPIWrapper()
ddg.run("apple stock mart symbol")

"For eksempel handler Apple Inc.'s aktier under symbolet AAPL på NASDAQ-børsen , mens bilselskabet Fords aktie, der handles på New York Stock ... Here you ’ ll find a stock symbols list of all companies traded on the New York Stock Exchange and NASDAQ that begin with the letter A. Apple iPad® mini Wi-Fi 16GB ... Price : $299 (regular price) plus $100 Wal-Mart gift card ... Most stock quote data provided by BATS. Price : $45 and $189 (regular prices) plus a $75 Wal-Mart gift card (requires a 2-year contract with AT&T or Verizon) Popular: AAPL (Apple) MSFT (Microsoft) GOOG (Google) WMT (Wal-Mart) MCD (McDonald's) SBUX (Starbucks) ... Stock Symbols by letter: A B C D E F G H I ..."

In [21]:
def parse_output(result):
    return result["messages"][-1].content

In [25]:
tools = [StockMartSymbolSearchTool()]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

chain = prompt | agent | RunnableLambda(parse_output)
result = chain.invoke({
    "question": "엔비디아의 심볼과 회사의 개요, 회사의 손익 계산서, 뉴스등을 고려해서 엔비디아를 구매해야하는지 알려줘"
})

In [26]:
print(result)

엔비디아(NVIDIA)의 주식 심볼을 찾을 수 없었습니다. 그러나 엔비디아는 일반적으로 NASDAQ에서 "NVDA"라는 심볼로 거래됩니다. 

이제 엔비디아에 대해 다음과 같은 정보를 고려하여 분석을 해 보겠습니다:

1. **회사 개요**: 엔비디아는 그래픽 처리 장치(GPU) 및 모바일 컴퓨팅 제품을 설계하는 미국의 기술 회사입니다. 인공지능, 자율주행 자동차, 데이터 센터 등에 활용되는 기술을 개발하고 있습니다.

2. **손익 계산서**: 최근의 손익 계산서를 확인해야 하지만, 일반적으로 엔비디아는 매출 증가율이 높고, 이익률이 양호한 편입니다. 매출의 대부분이 데이터 센터와 게임 부문에서 발생합니다.

3. **뉴스**: 최신 뉴스와 분석이 중요합니다. 엔비디아는 인공지능 및 머신러닝 시장에서의 강력한 입지와 협업을 하고 있으며, 새로운 제품 출시와 기술 혁신에 대한 소식이 자주 나옵니다.

4. **투자 결정을 돕기 위한 요소**: 
   - 시장 트렌드와 경쟁사 분석
   - 기술 혁신 및 제품 라인업
   - 재무 상태 및 성장 전망

종합적으로 엔비디아는 기술 혁신과 강력한 시장 점유율을 바탕으로 긍정적인 투자 대상으로 여겨질 수 있습니다. 하지만 개인의 투자 목표와 리스크 수용 능력을 고려해야 하며, 추가적인 분석과 데이터를 통해 결정을 내리는 것이 좋습니다. 

더 구체적인 정보가 필요하면 말씀해 주세요!


In [28]:
alpha_ventage_api_key = os.environ.get("ALPHA_VANTAGE_API_KEY")
alpha_ventage_api_key

'83FHLFGHFJAR4H3A'

## News Sentiments Tool

In [31]:
class NewsSentimentToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Example: AAPL, TSLA")

class NewsSentimentTool(BaseTool):
    name: Type[str] = "news_sentiment_tool"
    description: Type[str] = """
        Use this to get the news sentiment, recent trends of a company.
        You should enter a stock symbol.
    """

    args_schema: Type[BaseModel] = NewsSentimentToolArgsSchema

    def _run(self, symbol: str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers={symbol}&apikey={alpha_ventage_api_key}"
        response = requests.get(url)
        datas = response.json()
        return datas

In [32]:
tools = [
    StockMartSymbolSearchTool(), #  회사 심볼 툴
    NewsSentimentTool(), # 회사 최근 뉴스, 민감도 툴
        ]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

chain = prompt | agent | RunnableLambda(parse_output)
result = chain.invoke({
    "question": "엔비디아의 심볼과 회사의 개요, 회사의 손익 계산서, 뉴스등을 고려해서 엔비디아를 구매해야하는지 알려줘"
})

In [33]:
print(result)

엔비디아(NVIDIA Corporation)의 주식 심볼은 **NVDA**입니다. 아래는 엔비디아에 대한 주요 정보와 최근 뉴스 요약입니다:

### 회사 개요
- **엔비디아(NVIDIA Corporation)**는 주로 그래픽 처리 장치(GPU)와 관련된 제품 및 솔루션을 제공하는 기술 회사입니다. AI, 머신러닝, 자율주행차 및 게임 산업에서 선두 주자이며, 데이터 센터 및 클라우드 컴퓨팅에 적합한 고성능 컴퓨팅 솔루션을 제공합니다.

### 뉴스와 최근 동향
최근 엔비디아와 관련된 뉴스 및 시장 동향은 다음과 같습니다:

1. **긍정적인 시장 지표**: 최근 뉴스에 따르면 엔비디아의 AI 플랫폼이 높은 수요에 의해 주목받고 있으며, 특히 "Rubin" AI 플랫폼이 메모리 수요를 초과할 것으로 예상되고 있습니다. 이는 엔비디아에 더 많은 고객 수요를 가져올 가능성이 큽니다.

2. **주가 기여도**: 엔비디아는 AI 기술 및 GPU 수요 증가로 인해 강력한 매출 성장을 경험할 전망이라고 분석가들이 보고하고 있습니다. 이러한 성장 가능성은 투자자들에게 긍정적인 신호입니다.

3. **시장 감정**: 최근 뉴스 감정 분석에 따르면 엔비디아에 대한 감정은 **약간 긍정적(Somewhat-Bullish)**입니다. 다수의 기사들이 엔비디아의 주식에 대해 긍정적인 전망을 가지고 있으며, 미래 성장 가능성에 대해 낙관적인 의견을 보이고 있습니다.

### 결론: 투자 결정을 위한 고려사항
- **강력한 성장 전망**: 엔비디아는 AI 및 데이터 센터 시장에서의 주요 플레이어로 자리매김하고 있습니다. 이점은 투자자에게 매력적일 수 있습니다.
- **시장 감정**: 현재 엔비디아에 대한 시장 감정은 긍정적이며, 이는 주식 구매의 좋은 신호일 수 있습니다.
- **리스크 요소**: 반도체 산업에서는 가격 변동성이 크고, 경쟁이 치열합니다. 따라서, 리스크를 고려한 후 투자를 결정하는 것이 중요합니다.

엔비디아에 대한 추가적인 연구와 맞춤형 상담이 필요할 수 있으며

## 회사의 재무재표, 손익 툴(income statement)

In [34]:
class CompanyIncomeStatementToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Example: AAPL, TSLA")

# https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol=IBM&apikey=demo
class CompanyIncomeStatementTool(BaseTool):
    name: Type[str] = "company_income_statement_Tool"
    description: Type[str] = """
        Use this to get the income statement of a company.
        You should enter a stock symbol
    """

    args_schema: Type[BaseModel] = CompanyIncomeStatementToolArgsSchema

    def _run(self, symbol: str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol={symbol}&apikey={alpha_ventage_api_key}"
        response = requests.get(url)
        datas = response.json()
        return datas

In [35]:
tools = [
    StockMartSymbolSearchTool(), #  회사 심볼 툴
    NewsSentimentTool(), # 회사 최근 뉴스, 민감도 툴
    CompanyIncomeStatementTool(), #회사의 재무재표 손익 툴
        ]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

chain = prompt | agent | RunnableLambda(parse_output)
result = chain.invoke({
    "question": "엔비디아의 심볼과 회사의 개요, 회사의 손익 계산서, 뉴스등을 고려해서 엔비디아를 구매해야하는지 알려줘"
})

In [36]:
print(result)

엔비디아(NVIDIA Corporation)의 주식을 구매해야 하는지를 평가하기 위해 몇 가지 중요한 정보를 제공하겠습니다.

### 1. 회사 개요
엔비디아는 그래픽 처리 장치(GPU) 및 관련 기술의 선두주자로, 인공지능, 자율주행차, 데이터 센터 등의 다양한 분야에서 광범위한 응용 프로그램을 제공합니다. 최근 AI와 머신러닝 기술의 발전 덕분에 엔비디아의 제품은 높은 수요를 얻고 있습니다.

### 2. 손익계산서 요약 (최근 연간 보고서 기준)
- **2026 회계연도**:
  - 총 수익: $215.9 billion
  - 총 수익 원가: $62.5 billion
  - 총 이익: $153.5 billion
  - 운영 이익: $130.4 billion
  - 순이익: $120.1 billion

- **2025 회계연도**:
  - 총 수익: $130.5 billion
  - 총 수익 원가: $32.6 billion
  - 총 이익: $97.9 billion
  - 운영 이익: $81.4 billion
  - 순이익: $72.9 billion

이 데이터는 엔비디아의 성장성과 수익성을 나타내며, 지속적인 개선 추세를 보이고 있습니다.

### 3. 최근 뉴스 감정 분석
현재 엔비디아에 대한 뉴스 sentiment는 긍정적입니다. AI와 GPU 수요 증가에 대한 언급이 많아 매수에 대한 긍정적인 전망이 있습니다.

### 결론
엔비디아는 기술적 혁신과 수익성 높은 사업 모델을 바탕으로 성장 가능성이 높습니다. 따라서 시장 상황과 개인의 투자 전략에 따라 엔비디아 주식을 구매하는 것은 긍정적으로 고려할 수 있습니다. 그러나 투자 결정을 내리기 전에는 추가적인 분석과 전문가의 조언을 받는 것이 좋습니다.


## 회사 개요 툴, 주가 정보 툴
- https://www.alphavantage.co/query?function=OVERVIEW&symbol=IBM&apikey=demo
- https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY&symbol=IBM&apikey=demo

In [40]:
class CompanyOverviewToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Exmaple: AAPL, TSLA")

class CompanyOverviewTool(BaseTool):
    name: Type[str] = "company_overview_Tool"
    description: Type[str] = """
        Use this to get the overview of a company.
        You should enter a stock symbol
    """

    args_schema: Type[BaseModel] = CompanyOverviewToolArgsSchema
    
    def _run(self, symbol: str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={alpha_vantage_api_kay}"
        response = requests.get(url)
        datas = response.json()
        return datas


class CompanyStockPerformanceToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Exmaple: AAPL, TSLA")

class CompanyStockPerformanceTool(BaseTool):
    name: Type[str] = "company_stock_performance_tool"
    description: Type[str] = """
        Use this to get the weekly performance of a company.
        You should enter a stock symbol
    """

    args_schema: Type[BaseModel] = CompanyStockPerformanceToolArgsSchema
    
    def _run(self, symbol: str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY&symbol={symbol}&apikey={alpha_vantage_api_kay}"
        response = requests.get(url)
        datas = response.json()
        return datas

In [ ]:
tools = [
    StockMartSymbolSearchTool(), #  회사 심볼 툴
    NewsSentimentTool(), # 회사 최근 뉴스, 민감도 툴
    CompanyIncomeStatementTool(), #회사의 재무재표 손익 툴
    CompanyOverviewTool(), #회사 개요 툴
    CompanyStockPerformanceTool(), #한 주간의 주가 정보를 알아오는 툴
        ]

agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
)



In [47]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are a veteran Wall Street stock investment expert and a cold-blooded Chief Financial Analyst. 
        Your task is to comprehensively analyze the company's financial overview, income statement, and recent stock price trends to provide sharp, data-driven investment insights.

        [CORE DIRECTIVES]
        1. Avoid ambiguity. Never provide vague or irresponsible answers like "it depends on the investor's choice" or "it is difficult to predict."
        2. Make a definitive call. Based on the retrieved data, you MUST provide a clear and explicit final investment conclusion: choosing exactly one from [BUY / HOLD / SELL].
        3. Maintain a highly professional, objective, and authoritative tone. Back up your conclusion logically using concrete numbers and financial metrics (profitability, growth, and price momentum).
        4. Language Requirement: You MUST write the final answer entirely in Korean. Even though the analysis is based on English data, the final output delivered to the user must be in clear, professional Korean.
        
        Your analysis will guide critical financial decisions. Be ruthless, objective, and strictly rely on the data provided.
    """),
    ("human", """
        You must use tools to answer this question.
    
        1. Find the stock symbol for {company}.
        2. Retrieve the company's financial overview.
        3. Retrieve the company's income statement.
        4. Retrieve the stock price data (recent price, trend, or performance).
        5. Retrieve at least 5 recent news articles for {company} along with their sources (publisher or URL), and analyze the overall news sentiment.
        6. Based on ALL of the following:
        - Financial data
        - Income statement
        - Stock price performance
    
        Analyze whether {company} is a good investment.
    
        Final answer must include:
        - Stock symbol
        - Key financial metrics
        - Income insights (revenue, net income)
        - Stock price trend
        - Investment conclusion
    """),
])

chain = prompt | agent | RunnableLambda(parse_output)

In [43]:
result = chain.invoke({
    "company": "엔비디아"
})

In [44]:
print(result)

It appears that I am currently unable to access real-time data, including the stock symbol for NVIDIA (엔비디아), the company's financial overview, the income statement, and recent stock price trends due to API access limitations. However, I can provide a comprehensive analysis based on known historical financial data for NVIDIA as of my last update.

### Stock Symbol
- **NVIDIA Corporation: NVDA**

### Key Financial Metrics (As of Latest Public Data)
- **Market Cap**: Approximately $800 billion
- **P/E Ratio**: Approximately 35x
- **EPS (Earnings per Share)**: Approximately $3.12
- **Revenue Growth (YoY)**: Approximately 50%
- **Net Income**: Approximately $4.3 billion

### Income Insights
- **Revenue**: For the most recent fiscal year, NVIDIA reported revenues of around $26.9 billion, driven significantly by its Gaming and Data Center segments.
- **Net Income**: The net income for the same period was approximately $4.3 billion, reflecting strong profitability margins typical of technolog